## Resource efficient Hybrid RAG:
This file includes the configurations needed to run a resource  efficient RAG chatbot using a Small Language Model (SLM). Technologies and methodologies used:


*   Ollama: Configurations to run Ollama in a notebook / google colab
*   Small Language Model : Microsoft Phi
*   Hybrid search : Combines vector serach and knowledge graph
*   Chroma DB : For storing and retrieving embeddings







In [1]:
!sudo apt-get update && sudo apt-get install -y zstd

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,764 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-

In [2]:
# Download and install the Ollama Linux binary
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# Install the official Python library to interact with the Ollama server
!pip install -q ollama langchain langchain-community langchain-experimental langchain-ollama networkx
!pip install -q LLMGraphTransformer chromadb
!pip install -q matplotlib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━

In [4]:
import subprocess
import time
import socket

# Start the Ollama server process in the background
print("Launching Ollama background server...")
server_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Wait and verify until the server port (11434) becomes responsive
def wait_for_server(port=11434, timeout=30):
    start_time = time.time()
    while time.time() - start_time < timeout:
        try:
            with socket.create_connection(("127.0.0.1", port), timeout=1):
                print("Ollama server successfully initialized and active!")
                return True
        except OSError:
            time.sleep(1)
    print("Server initialization timed out.")
    return False

wait_for_server()

Launching Ollama background server...
Ollama server successfully initialized and active!


True

In [5]:
# Pull the target model
!ollama pull phi3

In [6]:
text = """
Albert Einstein[a] (14 March 1879 – 18 April 1955) was a German-born theoretical physicist who is best known for developing the theory of relativity. Einstein also made important contributions to quantum mechanics.[1][5] His mass–energy equivalence formula E = mc2, which arises from special relativity, has been called "the world's most famous equation".[6] He received the 1921 Nobel Prize in Physics for his services to theoretical physics, and especially for his discovery of the law of the photoelectric effect.[7]

Born in the German Empire, Einstein moved to Switzerland in 1895, forsaking his German citizenship (as a subject of the Kingdom of Württemberg)[note 1] the following year. In 1897, at the age of seventeen, he enrolled in the mathematics and physics teaching diploma program at the Swiss federal polytechnic school in Zurich, graduating in 1900. He acquired Swiss citizenship a year later, which he kept for the rest of his life, and afterwards secured a permanent position at the Swiss Patent Office in Bern. In 1905, he submitted a successful PhD dissertation to the University of Zurich. In 1914, he moved to Berlin to join the Prussian Academy of Sciences and the Humboldt University of Berlin, becoming director of the Kaiser Wilhelm Institute for Physics in 1917; he also became a German citizen again, this time as a subject of the Kingdom of Prussia.[note 1] In 1933, while Einstein was visiting the United States, Adolf Hitler came to power in Germany. Horrified by the Nazi persecution of his fellow Jews,[8] he decided to remain in the US, and was granted American citizenship in 1940.[9] On the eve of World War II, he endorsed a letter to President Franklin D. Roosevelt alerting him to the potential German nuclear weapons program and recommending that the US begin similar research.

In 1905, sometimes described as his annus mirabilis (miracle year), he published four groundbreaking papers.[10] In them, he outlined a theory of the photoelectric effect, explained Brownian motion, introduced his special theory of relativity, and demonstrated that if the special theory is correct, mass and energy are equivalent to each other. In 1915, he proposed a general theory of relativity that extended his system of mechanics to incorporate gravitation. A cosmological paper that he published the following year laid out the implications of general relativity for the modeling of the structure and evolution of the universe as a whole.[11][12] In 1917, Einstein wrote a paper which introduced the concepts of spontaneous emission and stimulated emission, the latter of which is the core mechanism behind the laser and maser, and which contained a trove of information that would be beneficial to developments in physics later on, such as quantum electrodynamics and quantum optics.[13]

In the middle part of his career, Einstein made important contributions to statistical mechanics and quantum theory. Especially notable was his work on the quantum physics of radiation, in which light consists of particles, subsequently called photons. With physicist Satyendra Nath Bose, he laid the groundwork for Bose–Einstein statistics. For much of the last phase of his academic life, Einstein worked on two endeavors that ultimately proved unsuccessful. First, he advocated against quantum theory's introduction of fundamental randomness into science's picture of the world, objecting that God does not play dice.[14] Second, he attempted to devise a unified field theory by generalizing his geometric theory of gravitation to include electromagnetism. As a result, he became increasingly isolated from mainstream modern physics.

In 1926, Einstein and his former student Leó Szilárd co-invented (and in 1930, patented) the Einstein refrigerator. This absorption refrigerator was then revolutionary for having no moving parts and using only heat as an input.[315] On 11 November 1930, U.S. patent 1,781,541 was awarded to Einstein and Leó Szilárd for the refrigerator. Their invention was not immediately put into commercial production, but the most promising of their patents were acquired by the Swedish company Electrolux.[note 6]

Einstein also invented an electromagnetic pump, sound reproduction device,[318] and several other household devices.
"""

In [7]:
#****************************************************#
# HYBRID GRAPH RAG USING PHI-3 + CHROMADB + NETWORKX #
#****************************************************#

import chromadb
import ollama
import networkx as nx
import json
import re

# STEP 1 - CREATE VECTOR DATABASE

print("Initializing ChromaDB...")

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection(
        name="knowledge_base"
    )
except:
    pass

collection = chroma_client.create_collection(
    name="knowledge_base"
)

Initializing ChromaDB...


In [8]:
# STEP 2 - CHUNK DOCUMENT

chunks = [
    chunk.strip()
    for chunk in text.split("\n\n")
    if chunk.strip()
]

print(f"Found {len(chunks)} chunks")

Found 6 chunks


In [9]:
# STEP 3 - STORE CHUNKS

for i, chunk in enumerate(chunks):

    collection.add(
        ids=[f"chunk_{i}"],
        documents=[chunk]
    )

print("Vector database ready")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 36.5MiB/s]


Vector database ready


In [10]:
# STEP 4 - EXTRACT KNOWLEDGE GRAPH

print("Generating knowledge graph...")

graph_prompt = f"""
You are an information extraction system.

Extract knowledge triples from the text.

Return ONLY JSON.

Format:

[
 {{
   "subject":"...",
   "relation":"...",
   "object":"..."
 }}
]

TEXT:

{text}
"""

response = ollama.chat(
    model="phi3",
    messages=[
        {
            "role":"user",
            "content":graph_prompt
        }
    ]
)

content = response["message"]["content"]

match = re.search(
    r"\[.*\]",
    content,
    re.DOTALL
)

if not match:
    raise Exception(
        "Could not parse graph triples"
    )

triples = json.loads(
    match.group()
)

print(
    f"Extracted {len(triples)} triples"
)

Generating knowledge graph...
Extracted 8 triples


In [11]:
# STEP 5 - BUILD NETWORKX GRAPH

G = nx.DiGraph()

for triple in triples:

    subject = triple.get("subject")
    relation = triple.get("relation")
    obj = triple.get("object")

    if not (
        subject
        and relation
        and obj
    ):
        continue

    G.add_node(subject)
    G.add_node(obj)

    G.add_edge(
        subject,
        obj,
        relation=relation
    )

print(
    f"Graph Nodes: {len(G.nodes())}"
)

print(
    f"Graph Edges: {len(G.edges())}"
)

Graph Nodes: 5
Graph Edges: 3


In [12]:
# STEP 6 - ENTITY EXTRACTION

def extract_entities(question):

    prompt = f"""
Extract important entities.

Return ONLY JSON.

Example:

{{
    "entities":[
        "Albert Einstein",
        "Germany"
    ]
}}

Question:

{question}
"""

    response = ollama.chat(
        model="phi3",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ]
    )

    content = response["message"]["content"]

    try:

        match = re.search(
            r"\{.*\}",
            content,
            re.DOTALL
        )

        result = json.loads(
            match.group()
        )

        return result.get(
            "entities",
            []
        )

    except:

        return []

In [13]:
# STEP 7 - VECTOR RETRIEVAL

def retrieve_vector_context(
    question,
    n_results=3
):

    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )

    docs = results["documents"][0]

    return "\n\n".join(docs)

In [14]:
# STEP 8 - GRAPH RETRIEVAL

def retrieve_graph_context(
    question,
    max_hops=2,
    max_facts=30
):

    entities = extract_entities(
        question
    )

    facts = set()

    for entity in entities:

        matching_nodes = []

        for node in G.nodes():

            if (
                entity.lower()
                in node.lower()
            ):
                matching_nodes.append(
                    node
                )

        for start_node in matching_nodes:

            visited = set()

            frontier = [start_node]

            for _ in range(max_hops):

                next_frontier = []

                for node in frontier:

                    if node in visited:
                        continue

                    visited.add(node)

                    for neighbor in G.neighbors(node):

                        relation = (
                            G[node]
                             [neighbor]
                             ["relation"]
                        )

                        facts.add(
                            f"{node} "
                            f"--{relation}--> "
                            f"{neighbor}"
                        )

                        next_frontier.append(
                            neighbor
                        )

                frontier = next_frontier

    if len(facts) == 0:

        for u, v, data in G.edges(
            data=True
        ):

            facts.add(
                f"{u} "
                f"--{data['relation']}--> "
                f"{v}"
            )

            if len(facts) >= max_facts:
                break

    return "\n".join(
        list(facts)[:max_facts]
    )

In [15]:
# STEP 9 - HYBRID RETRIEVAL

def hybrid_retrieval(question):

    vector_context = (
        retrieve_vector_context(
            question
        )
    )

    graph_context = (
        retrieve_graph_context(
            question
        )
    )

    return f"""
==============================
VECTOR RETRIEVAL
==============================

{vector_context}

==============================
GRAPH RETRIEVAL
==============================

{graph_context}
"""

In [16]:
# STEP 10 - HYBRID GRAPH RAG

def hybrid_graphrag(question):

    context = hybrid_retrieval(
        question
    )

    prompt = f"""
You are a helpful assistant.

Answer using BOTH:

1. Retrieved document chunks
2. Knowledge graph facts

Rules:

- Prefer document evidence.
- Use graph facts to infer relationships.
- Do not invent information.
- If answer is missing,
  say so.

CONTEXT:

{context}

QUESTION:

{question}

ANSWER:
"""

    stream = ollama.chat(
        model="phi3",
        messages=[
            {
                "role":"user",
                "content":prompt
            }
        ],
        stream=True
    )

    print("\nPhi-3:\n")

    for chunk in stream:

        print(
            chunk["message"]["content"],
            end="",
            flush=True
        )

    print(
        "\n"
        + "="*80
    )

In [17]:
# STEP 11 - CHATBOT

print(
    "\nHybrid GraphRAG Ready"
)

while True:

    question = input(
        "\nAsk a question "
        "(type exit to quit): "
    )

    if question.lower() in [
        "exit",
        "quit",
        "q"
    ]:
        break

    hybrid_graphrag(
        question
    )


Hybrid GraphRAG Ready

Ask a question (type exit to quit): What were the Einstein's inventions?

Phi-3:

**Retrieved Document Chunks:** According to a document, Albert Einstein not only developed theories such as special relativity and Brownian motion but also invented an electromagnetic pump (also known as the “Einstein refrigerator”), sound reproduction device, several other household devices, among others.

**Knowledge Graph Facts:** 
- Albert Einstein was born in Württemberg Kingdom of German Empire on March 14, 1879 [fact from 'Albert Einstein --born in the German Empire as a citizen of Württemberg Kingdom-->'] and later moved to Switzerland for citizenship. 
- In his "annus mirabilis" or miracle year (1905), he developed groundbreaking papers on special relativity explaining Brownian motion, introducing mass–energy equivalence formula E=mc^2 [fact from 'Albert Einstein --developed theory explaining Brownian motion and special relativity as groundbreaking papers from his annus mi